# 3. Feature Engineering

In this section, features were created to enable the model to understand the text data. 
Using the cleaned data (hazir_veri2.csv), the texts were vectorized using the TF-IDF method and 
combined with manually extracted statistical features. 

The TF-IDF vectors were stored in a sparse matrix structure to ensure memory efficiency. 
This approach aims to enable the model to learn more robustly and consistently by utilizing both the semantic content and structural features of the text. 
As a result, the X (features) and y (label) datasets were prepared for modeling.


In [20]:
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import hstack, csr_matrix

INPUT_CSV = "processed_reviews.csv"

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
print("Veri boyutu:", df.shape)
display(df.head())

required = ["Yorum", "clean_text", "label"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Eksik kolonlar: {missing}. Data_Prep çıktısı '{INPUT_CSV}' doğru mu?")

df["Yorum"] = df["Yorum"].fillna("").astype(str)
df["clean_text"] = df["clean_text"].fillna("").astype(str)

df["label"] = df["label"].astype(int)

print("Kolonlar tamam")
print("Label dağılımı:\n", df["label"].value_counts())

Veri boyutu: (2094, 4)


,Puan,Yorum,label,clean_text
0,"5,0",Hayatımda en çok içimi acıtan ağlata ağlata yü...,1.0,hayatımda içimi acıtan ağlata ağlata yüreğimi ...
1,"5,0",Şimdiye kadar Salya sümük ağladığım tek film t...,1.0,şimdiye kadar salya sümük ağladığım tek film t...
2,"5,0",Anlamadığım su yok ordan calinmis yok ona benz...,1.0,anlamadığım su yok ordan calinmis yok ona benz...
3,"5,0",Filmi dün izledim. Gitmeden önce hiçbir tavsiy...,1.0,filmi dün izledim gitmeden önce hiçbir tavsiye...
4,"4,5","Filmi ağlayarak mı izlemek ,az kalan bir cümle...",1.0,filmi ağlayarak izlemek kalan bir cümleiçiniz ...


Kolonlar tamam
Label dağılımı:
 label
1    1212
0     882
Name: count, dtype: int64


In [21]:
df["question_count"] = df["Yorum"].apply(lambda x: x.count("?"))

def upper_ratio(text: str) -> float:
    if not text:
        return 0.0
    upper_chars = sum(1 for c in text if c.isupper())
    return upper_chars / len(text)

df["upper_ratio"] = df["Yorum"].apply(upper_ratio)

def avg_word_len(text: str) -> float:
    words = text.split()
    if not words:
        return 0.0
    return sum(len(w) for w in words) / len(words)

df["avg_word"] = df["Yorum"].apply(avg_word_len)

positive_words = {"güzel", "iyi", "mükemmel", "çok"}
neg_words = {"değil", "yok", "hiç", "asla"}

def count_lexicon(text: str, lex: set) -> int:
    toks = text.lower().split()
    return sum(1 for t in toks if t in lex)

df["positive_count"] = df["Yorum"].apply(lambda x: count_lexicon(x, positive_words))
df["negation_count"] = df["Yorum"].apply(lambda x: count_lexicon(x, neg_words))

df["exclamation_count"] = df["Yorum"].apply(lambda x: x.count("!"))
df["char_count"] = df["Yorum"].apply(len)

custom_cols = [
    "question_count",
    "upper_ratio",
    "avg_word",
    "positive_count",
    "negation_count",
    "exclamation_count",
    "char_count",
]

print("Custom feature örnekleri:")
display(df[custom_cols].head())

Custom feature örnekleri:


,question_count,upper_ratio,avg_word,positive_count,negation_count,exclamation_count,char_count
0,0,0.012739,5.869565,2,0,0,157
1,0,0.041237,5.928571,0,0,0,97
2,0,0.002155,5.458333,0,5,0,464
3,1,0.017570,6.117857,5,0,0,1992
4,0,0.017007,6.972973,2,0,0,294


In [22]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5
)

X_tfidf = tfidf.fit_transform(df["clean_text"])
print("X_tfidf shape:", X_tfidf.shape)

# custom -> sparse
X_custom = df[custom_cols].astype(float).values
X_custom_sp = csr_matrix(X_custom)

X_final = hstack([X_tfidf, X_custom_sp]).tocsr()
y = df["label"].values

print("X_final shape:", X_final.shape)
print("y dağılımı:\n", pd.Series(y).value_counts())


X_tfidf shape: (2094, 4961)
X_final shape: (2094, 4968)
y dağılımı:
 1    1212
0     882
Name: count, dtype: int64


In [23]:
bow = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5
)
X_bow = bow.fit_transform(df["clean_text"])
print("X_bow shape:", X_bow.shape)

joblib.dump(X_bow, "X_bow.pkl")
joblib.dump(bow, "bow_vectorizer.pkl")
print("BoW kaydedildi: X_bow.pkl, bow_vectorizer.pkl")

X_bow shape: (2094, 4961)
BoW kaydedildi: X_bow.pkl, bow_vectorizer.pkl


In [24]:
try:
    from gensim.models import Word2Vec

    tokenized = df["clean_text"].str.split().tolist()

    w2v = Word2Vec(
        sentences=tokenized,
        vector_size=100,
        window=5,
        min_count=2,
        workers=4,
        sg=1,
        epochs=10
    )

    def doc_vector(tokens):
        vecs = [w2v.wv[w] for w in tokens if w in w2v.wv]
        if not vecs:
            return np.zeros(w2v.vector_size)
        return np.mean(vecs, axis=0)

    X_w2v = np.vstack([doc_vector(toks) for toks in tokenized])
    print("X_w2v shape:", X_w2v.shape)

    np.save("X_w2v.npy", X_w2v)
    w2v.save("word2vec.model")
    print("Word2Vec kaydedildi: X_w2v.npy, word2vec.model")

except Exception as e:
    print("Word2Vec kısmı atlandı (gensim/kurulum sorunu olabilir):", repr(e))

X_w2v shape: (2094, 100)
Word2Vec kaydedildi: X_w2v.npy, word2vec.model


In [25]:
df.to_csv("final_Data.csv", index=False, encoding="utf-8-sig")

joblib.dump(X_final, "X_features.pkl")
joblib.dump(y, "y_labels.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Kaydedildi: final_Data.csv, X_features.pkl, y_labels.pkl, tfidf_vectorizer.pkl")

Kaydedildi: final_Data.csv, X_features.pkl, y_labels.pkl, tfidf_vectorizer.pkl
